# Churn Model — XGBoost on the Gold Feature Table

EDA and training for the churn classifier.

Features come from the dbt-built gold table `analytics.customer_features` — this
notebook does **not** compute production features. It does model-specific prep,
trains XGBoost, tunes the decision threshold, evaluates, and (section 8)
explores candidate features. Shared logic is imported from `backend.ml`.

**Prerequisite:** the pipeline has been run (`docker compose up -d`, the loader,
the simulation, `dbt build`) so the gold table exists.

In [3]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score

from backend.db.session import engine
from backend.ml.data import load_gold, load_splits
from backend.ml.train import build_model, evaluate, tune_threshold, PARAMS

pd.set_option("display.max_columns", None)

ModuleNotFoundError: No module named 'pydantic_settings'

## 1. Load the gold table

In [ ]:
df = load_gold()

# numeric columns arrive from Postgres as Decimal — coerce for analysis
keys = {"customer_unique_id", "snapshot_date", "customer_state", "churned", "split"}
num = [c for c in df.columns if c not in keys]
df[num] = df[num].apply(pd.to_numeric)

print(f"{len(df):,} rows  -  {df.shape[1]} columns")
df.head()

## 2. Churn balance and the train/test/validation split

The `split` column is assigned deterministically in dbt and stratified by the
churn label — so every split carries the same churn rate.

In [ ]:
print("overall churn rate:", round(df["churned"].mean(), 3))

summary = df.groupby("split").agg(
    rows=("churned", "size"),
    churn_rate=("churned", "mean"),
).round(3)
display(summary)

summary["rows"].reindex(["train", "test", "validation"]).plot.bar(
    color="steelblue", title="Rows per split", rot=0)
plt.ylabel("rows"); plt.tight_layout(); plt.show()

## 3. Do the features separate churned from retained customers?

If the synthetic augmentation worked, retained customers should show better
first-order experiences (higher review score, faster delivery).

In [ ]:
signal = ["avg_review_score", "avg_delivery_days", "frequency",
          "recency_days", "monetary_avg", "order_interval_mean"]
df.groupby("churned")[signal].mean().round(2)

## 4. Prepare the splits and train

`load_splits()` reads the gold table, splits by the `split` column, and applies
leakage-free prep. `build_model()` sets `scale_pos_weight` from the class
balance.

In [ ]:
splits, feature_cols = load_splits()
X_train, y_train = splits["train"]
X_test,  y_test  = splits["test"]
X_val,   y_val   = splits["validation"]

print(f"features: {len(feature_cols)}  -  "
      f"train: {len(y_train):,}   test: {len(y_test):,}   val: {len(y_val):,}")

model = build_model(y_train)
model.fit(X_train, y_train)
print("scale_pos_weight:", round(model.scale_pos_weight, 3))

## 5. Tune the decision threshold, then evaluate

The model outputs probabilities; precision and recall depend on the cutoff. The
default 0.5 catches only ~67% of churners. We tune the threshold on the
**validation** PR curve — the highest-precision threshold that still catches
**85%** of churners — then evaluate at it.

(An F-beta objective collapses here: churn is the ~80% majority class, so
"predict everyone" maximises it. Test is held out from tuning, so its numbers
are unbiased.)

In [ ]:
threshold = tune_threshold(model, X_val, y_val)   # recall >= 0.85 on validation
print(f"tuned threshold: {threshold:.3f}   (default was 0.5)")

pd.DataFrame({
    "test @0.50":        evaluate(model, X_test, y_test, 0.5),
    "test @tuned":       evaluate(model, X_test, y_test, threshold),
    "validation @tuned": evaluate(model, X_val, y_val, threshold),
}).round(3)

## 6. ROC and precision-recall curves (validation)

The tuned operating point is marked on the PR curve; the dotted line is the
0.80 churn base rate (the precision a "predict everyone" classifier gets).

In [ ]:
proba = model.predict_proba(X_val)[:, 1]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))

fpr, tpr, _ = roc_curve(y_val, proba)
a1.plot(fpr, tpr); a1.plot([0, 1], [0, 1], "--", c="grey")
a1.set(title="ROC", xlabel="false positive rate", ylabel="true positive rate")

prec, rec, _ = precision_recall_curve(y_val, proba)
a2.plot(rec, prec)
op = evaluate(model, X_val, y_val, threshold)
a2.scatter([op["recall"]], [op["precision"]], color="crimson", zorder=5,
           label=f"tuned threshold = {threshold:.2f}")
a2.axhline(0.80, ls=":", c="grey", lw=1)
a2.legend()
a2.set(title="Precision-Recall", xlabel="recall", ylabel="precision")

plt.tight_layout(); plt.show()

## 7. Feature importance

A sanity check on the pipeline: the synthetic generator drove reorder behaviour
from review score and delivery speed, so those should dominate.

In [ ]:
imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
imp.plot.barh(figsize=(7, 6), color="steelblue",
              title="XGBoost feature importance")
plt.tight_layout(); plt.show()

## 8. Feature exploration — which candidate features are worth adding?

Before adding columns to the dbt gold model, test whether candidate features
actually separate churned from retained customers.

These candidates do not exist in the gold table yet, so they are computed here
**from the order-level data** (`int_customer_orders` joined to `stg_orders`),
using the same point-in-time cutoff the gold table uses (orders on or before
the snapshot date). This is exploration only — anything that earns its place
gets promoted to SQL in the dbt model.

In [ ]:
SNAPSHOT = pd.Timestamp("2017-08-01")

orders = pd.read_sql('''
    select co.customer_unique_id, co.order_id, co.order_status,
           co.order_purchase_timestamp, co.delivery_days, co.review_score,
           so.order_estimated_delivery_date, so.order_delivered_customer_date
    from analytics.int_customer_orders co
    join analytics.stg_orders so using (order_id)
''', engine)

# point-in-time: history on/before the snapshot, the cutoff the gold table uses
orders = orders[orders["order_purchase_timestamp"] <= SNAPSHOT].copy()
orders["days_late"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

print(f"{len(orders):,} historical orders  -  "
      f"{orders.customer_unique_id.nunique():,} customers")

In [ ]:
ordered = orders.sort_values("order_purchase_timestamp")

agg = ordered.groupby("customer_unique_id").agg(
    n_orders=("order_id", "size"),
    avg_days_late=("days_late", "mean"),
    late_rate=("days_late", lambda s: (s > 0).mean()),
    trouble_rate=("order_status", lambda s: (s != "delivered").mean()),
    review_rate=("review_score", lambda s: s.notna().mean()),
    first_review=("review_score", "first"),
    last_review=("review_score", "last"),
    first_delivery=("delivery_days", "first"),
    last_delivery=("delivery_days", "last"),
    first_ts=("order_purchase_timestamp", "first"),
    last_ts=("order_purchase_timestamp", "last"),
)
agg["review_trend"]   = agg["last_review"] - agg["first_review"]
agg["delivery_trend"] = agg["last_delivery"] - agg["first_delivery"]
agg["tenure_days"]    = (agg["last_ts"] - agg["first_ts"]).dt.days
agg["order_rate"]     = agg["n_orders"] / agg["tenure_days"].clip(lower=1)
agg.head()

In [ ]:
# join the gold churn label, then score each candidate's discriminative power
gold = df.set_index("customer_unique_id")
explore = agg.join(gold[["churned", "recency_days", "order_interval_median"]], how="inner")
explore["overdue_ratio"] = explore["recency_days"] / explore["order_interval_median"]

candidates = ["avg_days_late", "late_rate", "trouble_rate", "review_rate",
              "last_review", "review_trend", "last_delivery", "delivery_trend",
              "overdue_ratio", "order_rate"]

y = explore["churned"].astype(int)
rows = []
for c in candidates:
    s = pd.to_numeric(explore[c], errors="coerce")
    mask = s.notna() & np.isfinite(s)
    if mask.sum() < 50 or s[mask].nunique() < 2:
        continue
    auc = roc_auc_score(y[mask], s[mask])
    rows.append({
        "feature": c,
        "discriminative_AUC": round(max(auc, 1 - auc), 3),
        "mean_retained": round(float(s[mask & (y == 0)].mean()), 2),
        "mean_churned": round(float(s[mask & (y == 1)].mean()), 2),
        "coverage": round(float(mask.mean()), 2),
    })

pd.DataFrame(rows).sort_values("discriminative_AUC", ascending=False).reset_index(drop=True)

### What to do with this

`discriminative_AUC` is each candidate's single-feature, direction-agnostic
separation of churned vs retained (0.50 = no signal). Candidates clearly above
0.50 — and with reasonable `coverage` — earn a place in the **dbt gold model**:
added as SQL in `customer_features.sql` (and `int_customer_orders.sql` for
`days_late`, which needs the estimated delivery date). Candidates near 0.50, or
sparse, are not worth the column.

Feature engineering stays in dbt — this notebook only *tests* candidates.

## 9. Reproducible run and MLflow tracking

This notebook is for exploration. The **reproducible** training run lives in
`src/backend/ml/train.py`, which trains, tunes the threshold, logs
parameters/metrics/importance, and **registers** the model to `mlruns/`:

```
python -m backend.ml.train
```

Browse the tracked runs with `mlflow ui` from the repo root.